###Rag project

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import os

from google.colab import userdata
api_key = userdata.get('OpenAIKey')


In [ ]:
%pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv

In [25]:
!pip install -q langchain-huggingface sentence-transformers langchain-community faiss-cpu

In [26]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled

from langchain_openai import OpenAIEmbeddings, ChatOpenAI

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter
)

from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate



/tmp/ipykernel_18743/2186959762.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


#step 1a -indexing(Document INgestion)

In [ ]:
video_id = "NYFGCESmikA"

try:
    transcript_list = YouTubeTranscriptApi().fetch(
        video_id,
        languages=["en"]
    ).to_raw_data()

    print(transcript_list)

    transcript = " ".join(
        chunk["text"] for chunk in transcript_list
    )
    print(transcript)

except TranscriptsDisabled:
    print("Transcript is disabled or unavailable.")

[{'text': '- There are decades where nothing happens\nand weeks where decades happen.', 'start': 0.1, 'duration': 4.72}, {'text': 'And we have seen decades of progress', 'start': 5.78, 'duration': 3.78}, {'text': "happen in the last nine months.\nIf you're not recognizing the", 'start': 9.62, 'duration': 4.19}, {'text': "gravity of the moment, that's\nthe delusion. That's the", 'start': 14.9, 'duration': 3.74}, {'text': 'psychosis. Who would not get\ndelirious if suddenly a genie', 'start': 18.7, 'duration': 3.38}, {'text': 'pops out of the bottle and says,\n"You can have whatever you want.', 'start': 22.46, 'duration': 3.5}, {'text': "Every feature you've ever\ndreamed of in an operating", 'start': 26.3, 'duration': 3.63}, {'text': 'system, I can deliver them to you,', 'start': 30.04, 'duration': 1.48}, {'text': "most of them in five minutes, a few in\n20, and if we really go hog wild, it's", 'start': 32.439, 'duration': 3.981}, {'text': 'gonna take me two hours." I want\nthe fastest 

#step 1b indexing(text splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""])
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

362

In [ ]:
chunks[0]

Document(metadata={}, page_content='- There are decades where nothing happens\nand weeks where decades happen. And we have seen decades of progress happen in the last nine months.\nIf you\'re not recognizing the gravity of the moment, that\'s\nthe delusion. That\'s the psychosis. Who would not get\ndelirious if suddenly a genie pops out of the bottle and says,\n"You can have whatever you want. Every feature you\'ve ever\ndreamed of in an operating system, I can deliver them to you, most of them in five minutes, a few in\n20, and if we really go hog wild, it\'s gonna take me two hours." I want\nthe fastest car breaking the speed limits. I want the diver watch\nthat can go down the Mariana Trench. I want the operating system\nthat can install in less than 60 seconds. One of the things I always\nloved about race cars was when I would stumble out of the car\nabsolutely smashed and barely able to hold my head up, and I\'d lay down\non the garage floor and just think, "Holy fuck, I\'m alive.

#step 1c and 1d

In [28]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(chunks, embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [29]:
# Check the number of stored vectors
print("Stored vectors:", vector_store.index.ntotal)

# Check the embedding dimensions
print("Embedding dimensions:", vector_store.index.d)

# Check the number of document chunks
print("Document chunks:", len(vector_store.docstore._dict))

Stored vectors: 362
Embedding dimensions: 384
Document chunks: 362


##Step 2 Retrievel